In [ ]:
# from huggingface_hub import login
# login()
# !export HF_HUB_ENABLE_HF_TRANSFER=1

In [ ]:
import torch
from transformers import CsmForConditionalGeneration, AutoProcessor
from peft import PeftModel
import soundfile as sf
from IPython.display import Audio, display

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model and processor
base_model_id = "sesame/csm-1b"
adapter_model_id = "keanteng/sesame-csm-elise-lora"  # your uploaded model

# Load processor
# Load processor from local cache
processor = AutoProcessor.from_pretrained(
    base_model_id, 
    # local_files_only=True
)

# Load base model from local cache
base_model = CsmForConditionalGeneration.from_pretrained(
    base_model_id, 
    device_map=device,
    torch_dtype=torch.float16,
    # local_files_only=True
)

# Load adapter from local cache
model = PeftModel.from_pretrained(
    base_model, 
    adapter_model_id,
    local_files_only=True
)
model = model.merge_and_unload()  # Merge adapter weights into base model

# Optimize for generation
model.generation_config.max_length = 256
model.generation_config.use_cache = True
model.generation_config.cache_implementation = "static"

if hasattr(model, "depth_decoder"):
    model.depth_decoder.generation_config.cache_implementation = "static"


In [ ]:
# Define a simple input
conversation = [
    {"role": "0", "content": [
        {"type": "text", "text": "Hello! I'm so happy to see you today!"}
    ]},
]

# Process input
inputs = processor.apply_chat_template(
    conversation,
    tokenize=True,
    return_dict=True,
).to(device)

# Generate audio
audio = model.generate(**inputs, output_audio=True)

# Convert to numpy and save
audio_cpu = audio[0].to(torch.float32).cpu().numpy()
output_file = "output.wav"
sf.write(output_file, audio_cpu, 24000)

# Play audio if in notebook
try:
    display(Audio(output_file))
except:
    print(f"Audio saved to {output_file}")
